In [1]:
# ======================================================
# CELL 2: INITIALIZE SPARK
# ======================================================

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CSCI316-Baselines")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "50")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark Version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/12 20:25:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/12 20:25:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/12 20:25:31 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/02/12 20:25:31 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark Version: 4.1.1


In [2]:
# ======================================================
# CELL 3: EXTRACT DATA
# ======================================================

import os
import shutil

BASE_DIR = "/app"
DATA_PROCESSED = os.path.join(BASE_DIR, "data", "processed")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

# In Docker/Jupyter, we expect parquet to already exist in data/processed.
# If a zip exists (legacy), unpack it once.
zip_candidates = [
    os.path.join(DATA_PROCESSED, "valuation_ml_cv_ready.zip"),
    os.path.join(OUTPUTS_DIR, "artifacts", "valuation_ml_cv_ready.zip"),
    os.path.join(BASE_DIR, "valuation_ml_cv_ready.zip"),
]

zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)

if zip_path:
    print("Found zip:", zip_path)
    shutil.unpack_archive(zip_path, DATA_PROCESSED)
    print("Unzipped into:", DATA_PROCESSED)
else:
    print("No zip found (expected if parquet folder already exists).")

Found zip: /app/outputs/artifacts/valuation_ml_cv_ready.zip
Unzipped into: /app/data/processed


In [3]:
# ======================================================
# CELL 4: LOAD DATA
# ======================================================

import os
from pyspark.sql import functions as F

BASE_DIR = "/app"
DATA_PROCESSED = os.path.join(BASE_DIR, "data", "processed")

path = os.path.join(DATA_PROCESSED, "valuation_ml_cv_ready.parquet")

if not os.path.exists(path):
    raise FileNotFoundError(
        f"Could not find: {path}\n"
        "Run 03_cross_validation.ipynb first to generate valuation_ml_cv_ready.parquet."
    )

df = spark.read.parquet(path)

df = df.cache()
df.count()

print("Rows:", df.count())
print("Columns:", df.columns)

Rows: 88092
Columns: ['row_id', 'features', 'label', 'fold_id']


In [4]:
# ======================================================
# CELL 5: METRICS
# ======================================================

from pyspark.sql import Window


def compute_metrics(pred_df):

    total = pred_df.count()
    correct = pred_df.filter(F.col("label") == F.col("prediction")).count()

    accuracy = correct / total if total > 0 else 0.0

    tp = (
        pred_df
        .filter(F.col("label") == F.col("prediction"))
        .groupBy("label")
        .count()
        .withColumnRenamed("label", "cls")
        .withColumnRenamed("count", "tp")
    )

    pred_counts = (
        pred_df
        .groupBy("prediction")
        .count()
        .withColumnRenamed("prediction", "cls")
        .withColumnRenamed("count", "pred_total")
    )

    actual_counts = (
        pred_df
        .groupBy("label")
        .count()
        .withColumnRenamed("label", "cls")
        .withColumnRenamed("count", "actual_total")
    )

    per = (
        actual_counts
        .join(pred_counts, "cls", "full")
        .join(tp, "cls", "left")
        .na.fill(0)
    )

    per = (
        per
        .withColumn("fp", F.col("pred_total") - F.col("tp"))
        .withColumn("fn", F.col("actual_total") - F.col("tp"))
    )

    per = (
        per
        .withColumn(
            "precision",
            F.when(F.col("pred_total") == 0, 0.0)
             .otherwise(F.col("tp") / F.col("pred_total"))
        )
        .withColumn(
            "recall",
            F.when(F.col("actual_total") == 0, 0.0)
             .otherwise(F.col("tp") / F.col("actual_total"))
        )
    )

    per = per.withColumn(
        "f1",
        F.when(
            (F.col("precision") + F.col("recall")) == 0,
            0.0
        ).otherwise(
            2 * F.col("precision") * F.col("recall") /
            (F.col("precision") + F.col("recall"))
        )
    )

    rows = per.collect()

    return {
        "accuracy": accuracy,
        "precision": sum(r["precision"] for r in rows) / len(rows),
        "recall": sum(r["recall"] for r in rows) / len(rows),
        "f1": sum(r["f1"] for r in rows) / len(rows)
    }

In [5]:
# ======================================================
# CELL 6: LOGISTIC REGRESSION (CV)
# ======================================================

from pyspark.ml.classification import LogisticRegression
import pandas as pd
import time

folds = [r["fold_id"] for r in df.select("fold_id").distinct().orderBy("fold_id").collect()]

log_results = []

start = time.time()

for i, k in enumerate(folds, start=1):

    print(f"[LOGISTIC] Fold {i}/10 : {k}")

    train_df = df.filter(F.col("fold_id") != k).select("features", "label")
    test_df = df.filter(F.col("fold_id") == k).select("features", "label")

    lr = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=50,
        regParam=0.01,
        elasticNetParam=0.0
    )

    model = lr.fit(train_df)

    preds = model.transform(test_df).select("label", "prediction")

    m = compute_metrics(preds)
    m["fold"] = k

    log_results.append(m)

    print("   ", m)

log_df = pd.DataFrame(log_results)

print("Logistic done in", round((time.time()-start),2), "secs")

log_df


[LOGISTIC] Fold 1/10 : 0


    {'accuracy': 0.691338403905097, 'precision': 0.6861208030760135, 'recall': 0.6902892213588939, 'f1': 0.6877578582856906, 'fold': 0}
[LOGISTIC] Fold 2/10 : 1
    {'accuracy': 0.6905788876276958, 'precision': 0.6849969393899441, 'recall': 0.690960464366979, 'f1': 0.6869917708435191, 'fold': 1}
[LOGISTIC] Fold 3/10 : 2
    {'accuracy': 0.6914869466515323, 'precision': 0.6887345302303628, 'recall': 0.6917299407808026, 'f1': 0.6900470145225407, 'fold': 2}
[LOGISTIC] Fold 4/10 : 3
    {'accuracy': 0.6949710523328414, 'precision': 0.6890097462774887, 'recall': 0.6948361705869073, 'f1': 0.690999773146844, 'fold': 3}
[LOGISTIC] Fold 5/10 : 4
    {'accuracy': 0.6922465660120332, 'precision': 0.6892678292507003, 'recall': 0.691465218441968, 'f1': 0.6902599435672846, 'fold': 4}
[LOGISTIC] Fold 6/10 : 5
    {'accuracy': 0.690203201271427, 'precision': 0.6840321484306434, 'recall': 0.6883446316724412, 'f1': 0.6856914572294794, 'fold': 5}
[LOGISTIC] Fold 7/10 : 6
    {'accuracy': 0.68793279600408

,accuracy,precision,recall,f1,fold
0,0.691338,0.686121,0.690289,0.687758,0
1,0.690579,0.684997,0.690960,0.686992,1
2,0.691487,0.688735,0.691730,0.690047,2
3,0.694971,0.689010,0.694836,0.691000,3
4,0.692247,0.689268,0.691465,0.690260,4
5,0.690203,0.684032,0.688345,0.685691,5
6,0.687933,0.682557,0.689364,0.684631,6
7,0.692474,0.687123,0.692661,0.689096,7
8,0.699512,0.693084,0.699515,0.694933,8
9,0.686684,0.682105,0.687983,0.684130,9


In [6]:
# ======================================================
# CELL 7: DECISION TREE (CV)
# ======================================================

from pyspark.ml.classification import DecisionTreeClassifier

tree_results = []

start = time.time()

for i, k in enumerate(folds, start=1):

    print(f"[TREE] Fold {i}/10 : {k}")

    train_df = df.filter(F.col("fold_id") != k).select("features", "label")
    test_df = df.filter(F.col("fold_id") == k).select("features", "label")

    dt = DecisionTreeClassifier(
        featuresCol="features",
        labelCol="label",
        maxDepth=5,
        seed=42
    )

    model = dt.fit(train_df)

    preds = model.transform(test_df).select("label", "prediction")

    m = compute_metrics(preds)
    m["fold"] = k

    tree_results.append(m)

    print("   ", m)

tree_df = pd.DataFrame(tree_results)

print("Tree done in", round((time.time()-start),2), "secs")

tree_df


[TREE] Fold 1/10 : 0


    {'accuracy': 0.7609263253490748, 'precision': 0.7774393261358492, 'recall': 0.7606318182642934, 'f1': 0.7636884679011661, 'fold': 0}
[TREE] Fold 2/10 : 1


    {'accuracy': 0.7550510783200908, 'precision': 0.7687585319334364, 'recall': 0.7556220848804706, 'f1': 0.758563607569059, 'fold': 1}
[TREE] Fold 3/10 : 2


    {'accuracy': 0.7461975028376845, 'precision': 0.7650259865550781, 'recall': 0.7457210399310026, 'f1': 0.7500041473284024, 'fold': 2}
[TREE] Fold 4/10 : 3


    {'accuracy': 0.7604722442956068, 'precision': 0.7759010840813437, 'recall': 0.7612214080081198, 'f1': 0.7641680550807207, 'fold': 3}
[TREE] Fold 5/10 : 4


...

    {'accuracy': 0.7546827108638892, 'precision': 0.7713743598004257, 'recall': 0.7554648914360191, 'f1': 0.7586168809232189, 'fold': 6}
[TREE] Fold 8/10 : 7


    {'accuracy': 0.7570666363945965, 'precision': 0.7708950513732765, 'recall': 0.7564342060230599, 'f1': 0.7595538411622541, 'fold': 7}
[TREE] Fold 9/10 : 8


    {'accuracy': 0.752639346123283, 'precision': 0.7691423696535776, 'recall': 0.7524221470103324, 'f1': 0.7560530785919015, 'fold': 8}
[TREE] Fold 10/10 : 9


    {'accuracy': 0.7456010897945283, 'precision': 0.7640315677370393, 'recall': 0.7451977761898325, 'f1': 0.749857800580136, 'fold': 9}
Tree done in 67.37 secs


,accuracy,precision,recall,f1,fold
0,0.760926,0.777439,0.760632,0.763688,0
1,0.755051,0.768759,0.755622,0.758564,1
2,0.746198,0.765026,0.745721,0.750004,2
3,0.760472,0.775901,0.761221,0.764168,3
4,0.762516,0.762647,0.761481,0.761702,4
5,0.756045,0.771691,0.755818,0.759027,5
6,0.754683,0.771374,0.755465,0.758617,6
7,0.757067,0.770895,0.756434,0.759554,7
8,0.752639,0.769142,0.752422,0.756053,8
9,0.745601,0.764032,0.745198,0.749858,9


In [7]:
# ======================================================
# CELL 8: SAVE RESULTS
# ======================================================

import os

print("Logistic Average:")
print(log_df[["accuracy","precision","recall","f1"]].mean())

print("\nDecision Tree Average:")
print(tree_df[["accuracy","precision","recall","f1"]].mean())

BASE_DIR = "/app"
metrics_dir = os.path.join(BASE_DIR, "outputs", "metrics")
os.makedirs(metrics_dir, exist_ok=True)

log_path = os.path.join(metrics_dir, "logistic_results.csv")
tree_path = os.path.join(metrics_dir, "tree_results.csv")

log_df.to_csv(log_path, index=False)
tree_df.to_csv(tree_path, index=False)

print("\nSaved:")
print(" -", log_path)
print(" -", tree_path)

Logistic Average:
accuracy     0.691743
precision    0.686703
recall       0.691715
f1           0.688454
dtype: float64

Decision Tree Average:
accuracy     0.755120
precision    0.769691
recall       0.755001
f1           0.758123
dtype: float64

Saved:
 - /app/outputs/metrics/logistic_results.csv
 - /app/outputs/metrics/tree_results.csv


In [ ]:
# ======================================================
# CELL 9: LOGISTIC CONFUSION MATRIX
# ======================================================

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pdf = log_preds_df.select("label", "prediction").toPandas()

y_true = pdf["label"].astype(int)
y_pred = pdf["prediction"].astype(int)

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(values_format="d")

plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# ======================================================
# CELL 10: DECISION TREE CONFUSION MATRIX
# ======================================================

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pdf = tree_preds_df.select("label", "prediction").toPandas()

y_true = pdf["label"].astype(int)
y_pred = pdf["prediction"].astype(int)

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(values_format="d")

plt.title("Decision Tree Confusion Matrix")
plt.tight_layout()
plt.show()